# 10 — Comparaison complète : méthodes MKL, feature maps et classique

**Objectifs** :
1. Comparer les 7 méthodes MKL : Average, Centered, SDP, Projection, BO, Shapley, Single-Best
2. Introduire la feature map **EfficientSU2** (ansatz matériel-efficient)
3. Augmenter la taille des données (**N=200**, **Q=8 qubits**)
4. Comparer le QMKL aux classifieurs classiques (RBF-SVM, Random Forest, LR)
5. Identifier la méthode la plus **efficiente** (rapport performance / temps)

**Configuration** : N=200, Q=8, 16 kernels (12 standards + 4 EfficientSU2),
kernels calculés via Statevector (rapide, exact, pas de bruit de shot)

**Figures** (results/10/) :
1. F1 — Qualité des kernels individuels (KTA + AUC)
2. F2 — Comparaison des 7 méthodes MKL (20 runs)
3. F3 — Analyse des poids par méthode
4. F4 — Temps de calcul : efficience méthode
5. F5 — Quantum vs Classique
6. F6 — Tests statistiques (Wilcoxon)
7. F7 — Tableau de synthèse

In [2]:
import sys, os, warnings, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict
warnings.filterwarnings('ignore')

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

from src.mkl.alignment import centered_alignment, sdp_alignment, projection_alignment
from src.mkl.shapley_mkl import ShapleyMKL
from src.mkl.bayesian_optimizer import BayesianKernelOptimizer
from src.evaluation.statistical_analysis import multi_run_evaluation, summarize_multi_run

OUT = ROOT / 'results' / '10'; OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.titlesize': 13, 'savefig.bbox': 'tight'})
N_SAMPLES = 200; N_QUBITS = 8; N_RUNS = 20; SEED = 42; C_SVM = 1.0
print("=== Notebook 10 : Comparaison Complète ===")
print(f"N={N_SAMPLES}, Q={N_QUBITS}, {N_RUNS} runs")

ModuleNotFoundError: No module named 'src'

## 1. Chargement et prétraitement des données

In [ ]:
from data.loaders import load_dataset
from src.preprocessing import QuantumScaler, FeatureReducer

datasets_raw = {}
datasets_processed = {}

for ds_name, ds_key in [('German Credit', 'german_credit'),
                         ('Bank Marketing', 'bank_marketing')]:
    X_raw, y = load_dataset(ds_key, n_samples=N_SAMPLES, random_state=SEED)
    datasets_raw[ds_name] = (X_raw, y)

    reducer = FeatureReducer(n_components=N_QUBITS)
    scaler = QuantumScaler(feature_range=(0, 2))
    X_proc = scaler.fit_transform(reducer.fit_transform(X_raw))
    datasets_processed[ds_name] = (X_proc, y)

    print(f"{ds_name}: X={X_raw.shape} → PCA → {X_proc.shape}, "
          f"class balance={y.mean():.2f}")

## 2. Calcul des kernels quantiques (Statevector, exact)

**16 kernels** : 12 standards (Z, ZZ, Pauli variantes × 2 α) + 4 EfficientSU2 :
- EfficientSU2 : ansatz matériel-efficient Ry/Rz + CNOT, reps=1 et reps=2
- Kernel fidelity : K(x,x') = |⟨ψ(x')|ψ(x)⟩|² via produit scalaire de statevectors

In [ ]:
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.circuit.library import ZFeatureMap, ZZFeatureMap, PauliFeatureMap
from qiskit.quantum_info import Statevector

def build_su2_feature_map(n_qubits, alpha=1.0, reps=2, entanglement='linear'):
    """EfficientSU2-based feature map with data re-uploading."""
    x = ParameterVector('x', n_qubits)
    qc = QuantumCircuit(n_qubits)
    for rep in range(reps):
        for i in range(n_qubits):
            qc.ry(alpha * x[i], i)
            qc.rz(alpha * x[i], i)
        if entanglement == 'linear':
            for i in range(n_qubits - 1):
                qc.cx(i, i + 1)
        elif entanglement == 'circular':
            for i in range(n_qubits):
                qc.cx(i, (i + 1) % n_qubits)
    return qc

def compute_fidelity_kernel_sv(feature_map, X):
    """Compute fidelity kernel matrix via Statevector (exact, no shots noise)."""
    N = len(X)
    params = list(feature_map.parameters)
    # Precompute all statevectors
    svs = np.zeros((N, 2**feature_map.num_qubits), dtype=complex)
    for i, x in enumerate(X):
        param_dict = {p: float(v) for p, v in zip(params, x)}
        qc_bound = feature_map.assign_parameters(param_dict)
        svs[i] = Statevector.from_instruction(qc_bound).data
    # Vectorized fidelity kernel
    gram = svs @ svs.conj().T
    K = np.abs(gram) ** 2
    return K.real

# Build feature map library (use alpha= param instead of data_map_func for Qiskit 1.x compat)
Q = N_QUBITS
FM_LIBRARY = OrderedDict([
    ('Z α=1.0',       PauliFeatureMap(Q, reps=1, paulis=['Z'], alpha=1.0, entanglement='linear')),
    ('Z α=3.0',       PauliFeatureMap(Q, reps=1, paulis=['Z'], alpha=3.0, entanglement='linear')),
    ('ZZ α=1.0',      PauliFeatureMap(Q, reps=1, paulis=['Z','ZZ'], alpha=1.0, entanglement='linear')),
    ('ZZ α=4.0',      PauliFeatureMap(Q, reps=1, paulis=['Z','ZZ'], alpha=4.0, entanglement='linear')),
    ('Pauli α=0.6',   PauliFeatureMap(Q, reps=1, entanglement='linear', paulis=['Z','ZZ'], alpha=0.6)),
    ('Pauli α=3.0',   PauliFeatureMap(Q, reps=1, entanglement='linear', paulis=['Z','ZZ'], alpha=3.0)),
    ('P-XZ α=0.5',    PauliFeatureMap(Q, reps=1, entanglement='linear', paulis=['X','Z'], alpha=0.5)),
    ('P-XZ α=2.5',    PauliFeatureMap(Q, reps=1, entanglement='linear', paulis=['X','Z'], alpha=2.5)),
    ('P-YXX α=0.6',   PauliFeatureMap(Q, reps=1, entanglement='linear', paulis=['Y','XX'], alpha=0.6)),
    ('P-YXX α=3.0',   PauliFeatureMap(Q, reps=1, entanglement='linear', paulis=['Y','XX'], alpha=3.0)),
    ('P-YZX α=0.6',   PauliFeatureMap(Q, reps=1, entanglement='linear', paulis=['Y','ZX'], alpha=0.6)),
    ('P-YZX α=3.0',   PauliFeatureMap(Q, reps=1, entanglement='linear', paulis=['Y','ZX'], alpha=3.0)),
    ('SU2-r1 α=0.5',  build_su2_feature_map(Q, alpha=0.5, reps=1)),
    ('SU2-r1 α=2.0',  build_su2_feature_map(Q, alpha=2.0, reps=1)),
    ('SU2-r2 α=0.5',  build_su2_feature_map(Q, alpha=0.5, reps=2)),
    ('SU2-r2 α=2.0',  build_su2_feature_map(Q, alpha=2.0, reps=2)),
])

kernel_names = list(FM_LIBRARY.keys())
M = len(kernel_names)
print(f"Bibliothèque : {M} feature maps, Q={Q} qubits")

# Compute all kernels for all datasets
all_kernels = {}
for ds_name, (X, y) in datasets_processed.items():
    print(f"\nCalcul kernels pour {ds_name} (N={len(X)})...")
    K_list = []
    t0 = time.time()
    for fm_name, fm in FM_LIBRARY.items():
        t1 = time.time()
        K = compute_fidelity_kernel_sv(fm, X)
        K_list.append(K)
        print(f"  {fm_name:20s}: {time.time()-t1:.1f}s, "
              f"off_diag_mean={K[np.triu_indices(len(K),1)].mean():.4f}")
    elapsed = time.time()-t0
    all_kernels[ds_name] = K_list
    print(f"  Total: {elapsed:.1f}s ({M} kernels × {len(X)}×{len(X)})")

## Figure F1 — Qualité des kernels individuels

Pour chaque kernel, on mesure :
- **KTA** (Kernel-Target Alignment) : corrélation avec le kernel idéal K_y
- **AUC individuelle** : performance d'un SVM entraîné sur un seul kernel (5-fold CV)
- **Variance off-diag** : proxy inverse de la concentration (plus c'est varié, mieux c'est)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

def kta_score(K, y):
    Ky = (y[:,None]==y[None,:]).astype(float)
    num = np.sum(K*Ky); den = np.sqrt(np.sum(K*K)*np.sum(Ky*Ky)+1e-24)
    return num/den

def single_kernel_cv(K, y, cv=5, C=1.0):
    K_psd = K.copy()
    me = np.min(np.linalg.eigvalsh(K_psd))
    if me < 0: K_psd += (abs(me)+1e-8)*np.eye(K_psd.shape[0])
    svm = SVC(kernel='precomputed', C=C, probability=True)
    try:
        scores = cross_val_score(svm, K_psd, y, cv=cv, scoring='roc_auc')
        return np.mean(scores)
    except: return 0.5

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
colors_su2 = ['#e74c3c' if 'SU2' in n else '#3498db' for n in kernel_names]

for row, (ds_name, (X, y)) in enumerate(datasets_processed.items()):
    K_list = all_kernels[ds_name]

    # KTA
    kta_vals = [kta_score(K, y) for K in K_list]
    ax = axes[row, 0]
    bars = ax.barh(range(M), kta_vals, color=colors_su2, alpha=0.85)
    ax.set_yticks(range(M)); ax.set_yticklabels(kernel_names, fontsize=8)
    ax.set_xlabel('Kernel-Target Alignment')
    ax.set_title(f'{ds_name} — KTA', fontweight='bold')
    for i, v in enumerate(kta_vals):
        ax.text(v+0.001, i, f'{v:.3f}', va='center', fontsize=7)

    # Single-kernel AUC
    auc_vals = [single_kernel_cv(K, y, cv=5, C=C_SVM) for K in K_list]
    ax = axes[row, 1]
    bars = ax.barh(range(M), auc_vals, color=colors_su2, alpha=0.85)
    ax.set_yticks(range(M)); ax.set_yticklabels(kernel_names, fontsize=8)
    ax.set_xlabel('AUC (5-fold CV)')
    ax.set_title(f'{ds_name} — AUC individuelle', fontweight='bold')
    ax.axvline(0.5, color='grey', ls=':', label='Aléatoire')
    for i, v in enumerate(auc_vals):
        ax.text(v+0.002, i, f'{v:.3f}', va='center', fontsize=7)

    print(f"\n{ds_name} — Top 3 KTA: ", end='')
    top_kta = np.argsort(kta_vals)[::-1][:3]
    for i in top_kta: print(f"{kernel_names[i]}={kta_vals[i]:.3f}", end='  ')
    print(f"\n{ds_name} — Top 3 AUC: ", end='')
    top_auc = np.argsort(auc_vals)[::-1][:3]
    for i in top_auc: print(f"{kernel_names[i]}={auc_vals[i]:.3f}", end='  ')

from matplotlib.patches import Patch
axes[0,1].legend(handles=[Patch(color='#3498db',label='Standard'),
                           Patch(color='#e74c3c',label='EfficientSU2')],
                  fontsize=9, loc='lower right')

plt.suptitle("F1 — Qualité des 16 kernels individuels (KTA + AUC, N=200, Q=8)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT/'10_F1_kernel_quality.png', dpi=150); plt.show()
print("\n✓ F1")

## Figure F2 — Comparaison des 7 méthodes MKL (20 runs)

**Protocole IBM** : 20 splits aléatoires train/test (67/33), AUC comme métrique.

Méthodes :
- **Average** : w = 1/M
- **Single-Best** : meilleur kernel individuel (sélectionné par CV sur train)
- **Centered** : alignement centré (Cortes et al. 2012)
- **SDP** : programmation semi-définie (Lanckriet et al. 2004)
- **Projection** : alignement itératif glouton
- **BO** : optimisation bayésienne (BO-MKQSVM, 2025)
- **Shapley** : valeurs de Shapley Monte Carlo (200 permutations)

In [ ]:
def _build_tgt(y): return (y[:,None]==y[None,:]).astype(float)

def single_best_fn(K_list, y):
    """Select the single best kernel by CV on training data."""
    best_auc, best_idx = -1, 0
    for i, K in enumerate(K_list):
        auc = single_kernel_cv(K, y, cv=3, C=C_SVM)
        if auc > best_auc: best_auc, best_idx = auc, i
    w = np.zeros(len(K_list)); w[best_idx] = 1.0
    return w

def shapley_fn(K_list, y):
    shap = ShapleyMKL(scoring='roc_auc', n_cv=3, C=C_SVM, seed=SEED, verbose=False)
    shap.compute_shapley_montecarlo(K_list, y, n_permutations=200)
    return shap.get_weights()

def bo_fn(K_list, y):
    bo = BayesianKernelOptimizer(n_calls=30, n_initial_points=8,
                                  cv_folds=3, random_state=SEED, optimize_C=False)
    bo.optimize(K_list, y, scoring='roc_auc')
    w = np.array(bo.best_weights_)
    s = w.sum()
    return w/s if s>0 else np.ones(len(K_list))/len(K_list)

methods = OrderedDict([
    ('Average',     lambda Ks, y: np.ones(len(Ks))/len(Ks)),
    ('Single-Best', single_best_fn),
    ('Projection',  lambda Ks, y: projection_alignment(Ks, _build_tgt(y))),
    ('Centered',    lambda Ks, y: centered_alignment(Ks, _build_tgt(y))),
    ('SDP',         lambda Ks, y: sdp_alignment(Ks, _build_tgt(y))),
    ('BO',          bo_fn),
    ('Shapley-MC',  shapley_fn),
])

comparison_results = {}
method_times = {}

for ds_name, (X, y) in datasets_processed.items():
    K_list = all_kernels[ds_name]
    print(f"\n{'='*60}")
    print(f"{ds_name}: {M} kernels, N={len(y)}")
    print(f"{'='*60}")

    results = {}
    times   = {}

    for mname, mfn in methods.items():
        print(f"  {mname:15s}... ", end='', flush=True)
        t0 = time.time()
        try:
            res = multi_run_evaluation(
                kernel_matrices_full=K_list, y_full=y,
                methods={mname: mfn}, n_runs=N_RUNS,
                test_size=0.33, C=C_SVM, scoring='roc_auc'
            )
            results[mname] = res[mname]
        except Exception as e:
            print(f"ERREUR: {e}")
            results[mname] = [float('nan')] * N_RUNS
        elapsed = time.time()-t0
        times[mname] = elapsed
        mn = np.nanmean(results[mname])
        sd = np.nanstd(results[mname], ddof=1)
        print(f"{mn:.4f} ± {sd:.4f}  ({elapsed:.1f}s)")

    comparison_results[ds_name] = results
    method_times[ds_name] = times

In [ ]:
method_order = ['Average','Single-Best','Projection','Centered','SDP','BO','Shapley-MC']
colors_m = ['#95a5a6','#f39c12','#1abc9c','#e74c3c','#2ecc71','#9b59b6','#3498db']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (ds_name, results) in zip(axes, comparison_results.items()):
    avail = [m for m in method_order if m in results]
    data = [results[m] for m in avail]
    cols = [colors_m[method_order.index(m)] for m in avail]

    bp = ax.boxplot(data, labels=avail, patch_artist=True, widths=0.65)
    for patch, c in zip(bp['boxes'], cols):
        patch.set_facecolor(c); patch.set_alpha(0.75)

    means = [np.nanmean(results[m]) for m in avail]
    ax.scatter(range(1,len(avail)+1), means, marker='D', color='black', s=50, zorder=5)

    ax.set_ylabel('AUC (ROC)', fontsize=12)
    ax.set_title(ds_name, fontweight='bold', fontsize=14)
    ax.tick_params(axis='x', rotation=35)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(0.5, color='grey', ls=':', lw=1)

plt.suptitle("F2 — Comparaison des 7 méthodes MKL (20 runs, N=200, Q=8, M=16 kernels)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT/'10_F2_mkl_comparison.png', dpi=150); plt.show()
print("✓ F2")

## Figure F3 — Analyse des poids par méthode

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for row, (ds_name, (X, y)) in enumerate(datasets_processed.items()):
    K_list = all_kernels[ds_name]
    Kt = _build_tgt(y)

    weight_methods = OrderedDict([
        ('Centered', centered_alignment(K_list, Kt)),
        ('SDP',      sdp_alignment(K_list, Kt)),
        ('Projection', projection_alignment(K_list, Kt)),
    ])

    # Normalize all
    for k in weight_methods:
        w = weight_methods[k]; w = np.maximum(w, 0)
        s = w.sum(); weight_methods[k] = w/s if s>1e-12 else np.ones(M)/M

    # BO
    try:
        bo = BayesianKernelOptimizer(n_calls=30, n_initial_points=8,
                                      cv_folds=5, random_state=SEED, optimize_C=False)
        bo.optimize(K_list, y, scoring='roc_auc')
        w_bo = np.array(bo.best_weights_); s=w_bo.sum()
        weight_methods['BO'] = w_bo/s if s>0 else np.ones(M)/M
    except: weight_methods['BO'] = np.ones(M)/M

    # Shapley
    shap = ShapleyMKL(scoring='roc_auc', n_cv=5, C=C_SVM, seed=SEED, verbose=False)
    shap.compute_shapley_montecarlo(K_list, y, n_permutations=300)
    weight_methods['Shapley'] = shap.get_weights()

    for col, (mname, w) in enumerate(list(weight_methods.items())[:3]):
        ax = axes[row, col]
        colors_w = ['#e74c3c' if 'SU2' in n else '#3498db' for n in kernel_names]
        ax.barh(range(M), w, color=colors_w, alpha=0.85, edgecolor='white')
        ax.set_yticks(range(M))
        ax.set_yticklabels(kernel_names if col==0 else [], fontsize=8)
        ax.set_xlabel('Poids normalisé')
        ax.set_title(f'{ds_name} — {mname}', fontweight='bold')

    # Print top-3 weights for each method
    print(f"\n{ds_name}:")
    for mname, w in weight_methods.items():
        top3 = np.argsort(w)[::-1][:3]
        print(f"  {mname:12s}: " + ", ".join(
            f"{kernel_names[i]}={w[i]:.3f}" for i in top3))

plt.suptitle("F3 — Poids MKL par méthode (16 kernels, N=200, Q=8)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT/'10_F3_weight_analysis.png', dpi=150); plt.show()
print("✓ F3")

## Figure F4 — Efficience : rapport AUC / temps de calcul

Temps mesuré pour 20 runs complets (optimisation des poids + évaluation SVM).
L'efficience = AUC moyenne / log(temps), mesure le compromis performance-coût.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (ds_name, results) in zip(axes, comparison_results.items()):
    times = method_times[ds_name]
    avail = [m for m in method_order if m in results and m in times]
    auc_means = [np.nanmean(results[m]) for m in avail]
    time_vals = [times[m] for m in avail]

    cols = [colors_m[method_order.index(m)] for m in avail]

    # Scatter: AUC vs Time
    for i, (m, auc, t, c) in enumerate(zip(avail, auc_means, time_vals, cols)):
        ax.scatter(t, auc, color=c, s=200, zorder=5, edgecolor='black', lw=1.5)
        ax.annotate(m, (t, auc), textcoords="offset points",
                    xytext=(10, 5), fontsize=9, fontweight='bold')

    ax.set_xscale('log')
    ax.set_xlabel('Temps total (s, log)', fontsize=12)
    ax.set_ylabel('AUC moyenne', fontsize=12)
    ax.set_title(ds_name, fontweight='bold')
    ax.grid(alpha=0.3, which='both')

    # Efficiency frontier line
    sorted_pts = sorted(zip(time_vals, auc_means), key=lambda x: x[0])
    frontier_t, frontier_a = [sorted_pts[0][0]], [sorted_pts[0][1]]
    best_a = sorted_pts[0][1]
    for t, a in sorted_pts[1:]:
        if a > best_a: frontier_t.append(t); frontier_a.append(a); best_a = a
    ax.plot(frontier_t, frontier_a, 'k--', lw=1.5, alpha=0.5, label='Frontière efficiente')
    ax.legend(fontsize=9)

    print(f"\n{ds_name} — Temps (20 runs):")
    for m in avail:
        eff = np.nanmean(results[m]) / np.log10(times[m]+1)
        print(f"  {m:15s}: {times[m]:8.1f}s  AUC={np.nanmean(results[m]):.4f}  "
              f"efficience={eff:.4f}")

plt.suptitle("F4 — Compromis AUC / Temps de calcul (20 runs)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT/'10_F4_efficiency.png', dpi=150); plt.show()
print("✓ F4")

## Figure F5 — Quantum MKL vs Classifieurs classiques

Comparaison directe avec des baselines classiques standards, entraînés
sur les données brutes (pas de réduction PCA). C'est le benchmark
réaliste : le classique a accès à TOUTES les features.

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

classical_results = {}

for ds_name, (X_raw, y) in datasets_raw.items():
    print(f"\n{ds_name} — Classiques (20 runs):")
    classical = {}

    classifiers = OrderedDict([
        ('RBF-SVM',   make_pipeline(StandardScaler(), SVC(C=1.0, kernel='rbf', probability=True))),
        ('Linear-SVM', make_pipeline(StandardScaler(), SVC(C=1.0, kernel='linear', probability=True))),
        ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=SEED)),
        ('Gradient Boost', GradientBoostingClassifier(n_estimators=100, random_state=SEED)),
        ('Logistic Reg',  make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))),
    ])

    for clf_name, clf in classifiers.items():
        scores = []
        for run in range(N_RUNS):
            Xtr, Xte, ytr, yte = train_test_split(X_raw, y, test_size=0.33,
                                                     random_state=SEED+run, stratify=y)
            try:
                clf.fit(Xtr, ytr)
                if hasattr(clf, 'predict_proba'):
                    s = roc_auc_score(yte, clf.predict_proba(Xte)[:,1])
                else:
                    s = roc_auc_score(yte, clf.decision_function(Xte))
                scores.append(s)
            except: scores.append(float('nan'))
        classical[clf_name] = scores
        print(f"  {clf_name:20s}: {np.nanmean(scores):.4f} ± {np.nanstd(scores,ddof=1):.4f}")

    classical_results[ds_name] = classical

# Combined plot
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, ds_name in zip(axes, comparison_results.keys()):
    # Best 3 quantum methods + all classical
    qres = comparison_results[ds_name]
    cres = classical_results[ds_name]

    q_methods = sorted(qres.keys(), key=lambda m: -np.nanmean(qres[m]))[:4]
    all_methods = list(cres.keys()) + q_methods
    all_data = [cres[m] for m in cres.keys()] + [qres[m] for m in q_methods]
    all_colors = ['#f39c12']*len(cres) + ['#3498db']*len(q_methods)

    bp = ax.boxplot(all_data, labels=all_methods, patch_artist=True, widths=0.6)
    for patch, c in zip(bp['boxes'], all_colors):
        patch.set_facecolor(c); patch.set_alpha(0.75)

    means = [np.nanmean(d) for d in all_data]
    ax.scatter(range(1,len(all_methods)+1), means, marker='D', color='black', s=40, zorder=5)

    ax.set_ylabel('AUC'); ax.set_title(ds_name, fontweight='bold')
    ax.tick_params(axis='x', rotation=45); ax.grid(axis='y', alpha=0.3)

axes[0].legend(handles=[Patch(color='#f39c12',label='Classique'),
                         Patch(color='#3498db',label='Quantum MKL')],
                fontsize=10, loc='lower right')

plt.suptitle("F5 — Quantum MKL (meilleures méthodes) vs Classifieurs classiques",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT/'10_F5_quantum_vs_classical.png', dpi=150); plt.show()
print("✓ F5")

## Figure F6 — Tests statistiques (Wilcoxon signed-rank)

In [ ]:
from scipy.stats import wilcoxon

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, (ds_name, results) in zip(axes, comparison_results.items()):
    avail = [m for m in method_order if m in results]
    K_m = len(avail)
    pval_matrix = np.ones((K_m, K_m))
    win_matrix = np.zeros((K_m, K_m))

    for i in range(K_m):
        for j in range(K_m):
            if i == j: continue
            s_i = np.array(results[avail[i]])
            s_j = np.array(results[avail[j]])
            mask = ~(np.isnan(s_i) | np.isnan(s_j))
            if mask.sum() < 5: continue
            try:
                stat, pval = wilcoxon(s_i[mask], s_j[mask], alternative='greater')
                pval_matrix[i, j] = pval
                win_matrix[i, j] = np.mean(s_i[mask] > s_j[mask])
            except: pass

    # Show p-value matrix
    im = ax.imshow(-np.log10(pval_matrix + 1e-10), cmap='RdYlGn', aspect='auto',
                   vmin=0, vmax=3)
    ax.set_xticks(range(K_m)); ax.set_yticks(range(K_m))
    ax.set_xticklabels(avail, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(avail, fontsize=9)
    ax.set_title(f'{ds_name} — −log₁₀(p) [ligne > colonne]', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8, label='−log₁₀(p-value)')

    for i in range(K_m):
        for j in range(K_m):
            p = pval_matrix[i, j]
            sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
            ax.text(j, i, f'{p:.3f}\n{sig}', ha='center', va='center',
                    fontsize=7, color='white' if -np.log10(p+1e-10)>1.5 else 'black')

plt.suptitle("F6 — Tests de Wilcoxon (p < 0.05 = significatif)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT/'10_F6_wilcoxon_tests.png', dpi=150); plt.show()
print("✓ F6")

## Figure F7 — Tableau de synthèse

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
ax.axis('off')

ds_names = list(comparison_results.keys())
all_methods_display = list(method_order)
# Add best classical
for ds in ds_names:
    best_cls_name = max(classical_results[ds], key=lambda m: np.nanmean(classical_results[ds][m]))
    if best_cls_name not in all_methods_display:
        all_methods_display.append(best_cls_name)

col_labels = []
for ds in ds_names:
    col_labels += [f'{ds}\nAUC', '±std', 'Temps(s)', 'Rang']
col_labels.append('Rang\nmoyen')

table_data = []
for method in all_methods_display:
    row = []; ranks = []
    for ds in ds_names:
        if method in comparison_results[ds]:
            scores = comparison_results[ds][method]
            t = method_times[ds].get(method, 0)
        elif method in classical_results.get(ds, {}):
            scores = classical_results[ds][method]
            t = 0
        else:
            row += ['—','—','—','—']; continue

        mn, sd = np.nanmean(scores), np.nanstd(scores, ddof=1)

        # Rank among ALL methods (quantum + classical)
        all_m = {}
        for m in comparison_results.get(ds, {}):
            all_m[m] = np.nanmean(comparison_results[ds][m])
        for m in classical_results.get(ds, {}):
            all_m[m] = np.nanmean(classical_results[ds][m])
        sorted_m = sorted(all_m, key=lambda x: -all_m[x])
        rank = sorted_m.index(method)+1 if method in sorted_m else len(sorted_m)

        row += [f'{mn:.4f}', f'{sd:.4f}', f'{t:.0f}' if t>0 else '—', f'#{rank}']
        ranks.append(rank)

    row.append(f'{np.mean(ranks):.1f}' if ranks else '—')
    table_data.append(row)

table = ax.table(cellText=table_data, rowLabels=all_methods_display,
                 colLabels=col_labels, cellLoc='center', loc='center')
table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1.0, 1.8)

# Style headers
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Highlight best AUC per dataset
for d_idx in range(len(ds_names)):
    col_auc = d_idx * 4
    vals = []
    for i in range(len(all_methods_display)):
        try: vals.append((i, float(table_data[i][col_auc])))
        except: pass
    if vals:
        best_i = max(vals, key=lambda x: x[1])[0]
        table[best_i+1, col_auc].set_facecolor('#a8e6cf')

# Color classical row differently
for i, m in enumerate(all_methods_display):
    if m in ['RBF-SVM','Linear-SVM','Random Forest','Gradient Boost','Logistic Reg']:
        for j in range(len(col_labels)):
            if table[i+1, j].get_facecolor()[0]>0.9:
                table[i+1, j].set_facecolor('#fff3e0')

plt.title("F7 — Synthèse complète : Quantum MKL + Classique (N=200, Q=8, M=16)",
          fontsize=14, fontweight='bold', pad=20)
plt.savefig(OUT/'10_F7_synthesis_table.png', dpi=150, bbox_inches='tight')
plt.show(); print("✓ F7")

In [ ]:
print("="*70)
print("RÉSUMÉ COMPARAISON COMPLÈTE")
print("="*70)

for ds_name in comparison_results:
    print(f"\n{ds_name}:")
    qres = comparison_results[ds_name]
    cres = classical_results[ds_name]

    # Best quantum
    best_q_name = max(qres, key=lambda m: np.nanmean(qres[m]))
    best_q_auc = np.nanmean(qres[best_q_name])
    # Best classical
    best_c_name = max(cres, key=lambda m: np.nanmean(cres[m]))
    best_c_auc = np.nanmean(cres[best_c_name])

    print(f"  Meilleure méthode MKL: {best_q_name} (AUC={best_q_auc:.4f})")
    print(f"  Meilleur classique:    {best_c_name} (AUC={best_c_auc:.4f})")
    print(f"  Δ(quantum−classique) = {best_q_auc-best_c_auc:+.4f}")

    # Most efficient
    for m in method_order:
        if m in method_times[ds_name]:
            t = method_times[ds_name][m]
            a = np.nanmean(qres.get(m,[0]))
            print(f"    {m:15s}: AUC={a:.4f}, t={t:.0f}s")

print(f"\nFigures sauvegardées dans {OUT}")
print("="*70)